# Project MVP: Anatomical Discovery (Obermeyer Replication)
This notebook replicates the **AI Discovery** methodology of the Obermeyer et al. 2026 paper. 

### The Challenge:
Obermeyer uses a "low-cost" signal (**Lead I**) to predict a ground truth that is typically **invisible** on that signal (**Structural Echo/RWMA**). 

### Our Replication Script:
We use **Lead I** (Lateral) to predict **Inferior Myocardial Infarction** (IMI). 
- **Discovery Aspect**: A human cardiologist cannot reliably diagnose an IMI from Lead I alone; they require Leads II, III, and aVF. 
- **Workflow**: 
    1. **Pre-training (PTB-XL)**: Train the ResNet-18 to detect any MI (General Cardiac Features).
    2. **Discovery Task (PTB-XL)**: Evaluate/Fine-tune to detect specifically **Inferior MIs** using only Lead I.
    3. **Cross-Dataset Validation (Shaoxing)**: Evaluate if the "Discovery" generalizes to the Chinese population targeting "Lower Wall" MIs.

In [1]:
import os
import pandas as pd
import numpy as np
import wfdb
import ast
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
import optuna

# Set device
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: mps


## 1. Data Loading & Anatomical Labeling (PTB-XL)

In [2]:
ptb_path = '../data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/'

def load_ptbxl_data(path):
    df = pd.read_csv(os.path.join(path, 'ptbxl_database.csv'), index_col=0)
    df.scp_codes = df.scp_codes.apply(lambda x: ast.literal_eval(x))
    return df

def aggregate_diagnostic(y_dic, agg_df):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index: tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

agg_df = pd.read_csv(os.path.join(ptb_path, 'scp_statements.csv'), index_col=0)
agg_df_mi = agg_df[agg_df.diagnostic_class == 'MI']

df = load_ptbxl_data(ptb_path)

# Create Global MI and Inferior MI (IMI) labels
df['MI'] = df.scp_codes.apply(lambda x: 1 if any(k in agg_df_mi.index for k in x.keys()) else 0)
df['IMI'] = df.scp_codes.apply(lambda x: 1 if 'IMI' in x.keys() else 0)

print(f"Total samples: {len(df)}")
print(f"Global MI prevalence: {df.MI.mean()*100:.2f}%")
print(f"Inferior MI (Hidden Target) prevalence: {df.IMI.mean()*100:.2f}%")

Total samples: 21799
Global MI prevalence: 25.09%
Inferior MI (Hidden Target) prevalence: 12.28%


In [3]:
class PTBXLDataset(Dataset):
    def __init__(self, df, base_path, target='MI', lead_idx=0):
        self.df = df
        self.base_path = base_path
        self.target = target
        self.lead_idx = lead_idx
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        filename = os.path.join(self.base_path, self.df.iloc[idx].filename_lr)
        signal, meta = wfdb.rdsamp(filename)
        lead_signal = signal[:, self.lead_idx]
        lead_signal = (lead_signal - np.mean(lead_signal)) / (np.std(lead_signal) + 1e-8)
        label = self.df.iloc[idx][self.target]
        return torch.tensor(lead_signal, dtype=torch.float32).unsqueeze(0), torch.tensor(label, dtype=torch.float32)

train_df = df[df.strat_fold < 9]
val_df = df[df.strat_fold == 9]
test_df = df[df.strat_fold == 10]

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 17418, Val: 2183, Test: 2198


## 2. Shaoxing Discovery Dataset (Lower Wall MI)

In [4]:
shaoxing_path = '../data/a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0/'
IMI_CODE = '164865005' # Note: Shaoxing uses common code, but we search for synonyms

class ShaoxingDataset(Dataset):
    def __init__(self, base_path, lead_idx=0, limit=None):
        self.base_path = base_path
        self.lead_idx = lead_idx
        self.records = []
        records_file = os.path.join(base_path, 'RECORDS')
        with open(records_file, 'r') as f:
            subfolders = [line.strip() for line in f if line.strip()]
        
        for sub in subfolders:
            if limit and len(self.records) >= limit: break
            sub_path = os.path.join(base_path, sub)
            if os.path.exists(sub_path):
                files = [f.replace('.hea', '') for f in os.listdir(sub_path) if f.endswith('.hea')]
                for f in files:
                    self.records.append(os.path.join(sub, f))
                    if limit and len(self.records) >= limit: break
        if limit: self.records = self.records[:limit]
        print(f"Found {len(self.records)} Shaoxing records.")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec_path = os.path.join(self.base_path, self.records[idx])
        try:
            signal, meta = wfdb.rdsamp(rec_path)
            lead_signal = signal[::5, self.lead_idx]
            lead_signal = (lead_signal - np.mean(lead_signal)) / (np.std(lead_signal) + 1e-8)
            
            label = 0
            for comment in meta['comments']:
                if 'Dx:' in comment:
                    # We target general MI as a starting discovery point in Shaoxing
                    if IMI_CODE in comment: label = 1
                    break
            
            return torch.tensor(lead_signal, dtype=torch.float32).unsqueeze(0), torch.tensor(label, dtype=torch.float32)
        except Exception as e:
            return torch.tensor(np.zeros(1000), dtype=torch.float32).unsqueeze(0), torch.tensor(0, dtype=torch.float32)

shaoxing_dataset = ShaoxingDataset(shaoxing_path, limit=5000)

Found 5000 Shaoxing records.


## 3. Model Architecture: 1D ResNet-18

In [5]:
def conv3x1(in_planes, out_planes, stride=1):
    return nn.Conv1d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)

class BasicBlock1D(nn.Module):
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(BasicBlock1D, self).__init__()
        self.conv1 = conv3x1(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x1(planes, planes)
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None: residual = self.downsample(x)
        out += residual
        out = self.relu(out)
        return out

class ResNet1D(nn.Module):
    def __init__(self, block, layers, num_classes=1):
        self.inplanes = 64
        super(ResNet1D, self).__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.inplanes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(planes),
            )
        layers = [block(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes
        for _ in range(1, blocks): layers.append(block(self.inplanes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        return self.fc(x.view(x.size(0), -1))

def resnet18_1d(): return ResNet1D(BasicBlock1D, [2, 2, 2, 2])

## 4. Execution Engine (Train/Eval)

In [6]:
def evaluate(loader, model, criterion, device):
    model.eval()
    all_losses, all_preds, all_targets = [], [], []
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs).squeeze()
            if outputs.dim() == 0: outputs = outputs.unsqueeze(0)
            loss = criterion(outputs, targets)
            all_losses.append(loss.item())
            all_preds.extend(torch.sigmoid(outputs).cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
    if len(set(all_targets)) < 2: return np.mean(all_losses), 0.5
    return np.mean(all_losses), roc_auc_score(all_targets, all_preds)

def train_epoch(loader, model, criterion, optimizer, device, limit=None):
    model.train()
    losses, preds, targets = [], [], []
    for i, (inputs, labels) in enumerate(loader):
        if limit and i >= limit: break
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs).squeeze()
        if outputs.dim() == 0: outputs = outputs.unsqueeze(0)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        preds.extend(torch.sigmoid(outputs).detach().cpu().numpy())
        targets.extend(labels.cpu().numpy())
    if len(set(targets)) < 2: return np.mean(losses), 0.5
    return np.mean(losses), roc_auc_score(targets, preds)

## 5. Phase 1: Pre-training on General MI (Replicating Obermeyer Swedish Study)

In [7]:
def objective(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64])
    
    train_loader = DataLoader(PTBXLDataset(train_df, ptb_path, target='MI'), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(PTBXLDataset(val_df, ptb_path, target='MI'), batch_size=batch_size)
    
    model = resnet18_1d().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    
    # Fast subset run for optimization
    train_epoch(train_loader, model, criterion, optimizer, device, limit=30)
    _, val_auc = evaluate(val_loader, model, criterion, device)
    return val_auc

print("Finding best general MI detection parameters...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=3)
best_params = study.best_params

[I 2026-03-05 18:31:46,240] A new study created in memory with name: no-name-be1d563b-0bba-449e-a9f3-6a9427695d7f


Finding best general MI detection parameters...


[I 2026-03-05 18:31:53,306] Trial 0 finished with value: 0.7163285318184892 and parameters: {'lr': 0.0001967784631370497, 'batch_size': 32}. Best is trial 0 with value: 0.7163285318184892.


[I 2026-03-05 18:31:57,856] Trial 1 finished with value: 0.7108135524447149 and parameters: {'lr': 0.0001840993254250375, 'batch_size': 32}. Best is trial 0 with value: 0.7163285318184892.


[I 2026-03-05 18:32:02,429] Trial 2 finished with value: 0.7267723901625301 and parameters: {'lr': 5.43219152029178e-05, 'batch_size': 32}. Best is trial 2 with value: 0.7267723901625301.


In [8]:
print("Training Pre-trained Model on General MI...")
model = resnet18_1d().to(device)
optimizer = optim.Adam(model.parameters(), lr=best_params['lr'])
criterion = nn.BCEWithLogitsLoss()

train_loader = DataLoader(PTBXLDataset(train_df, ptb_path, target='MI'), batch_size=best_params['batch_size'], shuffle=True)
for epoch in range(2):
    l, a = train_epoch(train_loader, model, criterion, optimizer, device, limit=50)
    print(f"Pre-train Epoch {epoch+1}: Loss {l:.3f}, AUC {a:.3f}")

Training Pre-trained Model on General MI...


Pre-train Epoch 1: Loss 0.569, AUC 0.639


Pre-train Epoch 2: Loss 0.520, AUC 0.703


## 6. Phase 2: Anatomical Discovery (Inferior MI from Lead I)

In [9]:
print("Starting Discovery Task: Predicting Hidden Inferior MI from Lead I...")

# Set goal to IMI (Hidden from Lead I)
imi_train_loader = DataLoader(PTBXLDataset(train_df, ptb_path, target='IMI'), batch_size=32, shuffle=True)
imi_val_loader = DataLoader(PTBXLDataset(val_df, ptb_path, target='IMI'), batch_size=32)

# Fine-tune ONLY the head to look for hidden signals
for param in model.parameters(): param.requires_grad = False
for param in model.fc.parameters(): param.requires_grad = True
optimizer = optim.Adam(model.fc.parameters(), lr=5e-4)

for epoch in range(3):
    l, a = train_epoch(imi_train_loader, model, criterion, optimizer, device, limit=100)
    _, v_auc = evaluate(imi_val_loader, model, criterion, device)
    print(f"Discovery Epoch {epoch+1}: Loss {l:.3f} | Val AUC (Lead I -> IMI): {v_auc:.4f}")

Starting Discovery Task: Predicting Hidden Inferior MI from Lead I...


Discovery Epoch 1: Loss 0.363 | Val AUC (Lead I -> IMI): 0.6736


Discovery Epoch 2: Loss 0.347 | Val AUC (Lead I -> IMI): 0.6743


Discovery Epoch 3: Loss 0.346 | Val AUC (Lead I -> IMI): 0.6747


## 7. Cross-Population Discovery (Shaoxing Evaluation)
We now test if the model's ability to "discover" lower wall damage from lead I generalizes to China.

In [10]:
shao_loader = DataLoader(shaoxing_dataset, batch_size=32)
_, s_auc = evaluate(shao_loader, model, criterion, device)
print(f"Shaoxing Discovery (Zero-shot Lower Wall Detection from Lead I): AUC {s_auc:.4f}")

Shaoxing Discovery (Zero-shot Lower Wall Detection from Lead I): AUC 0.8326
